# Calculating Gravity Data
## Bringing in Minecraft block array

In [ ]:
import numpy as np
from numpy import random
import pandas as pd
import sys
import matplotlib.pyplot as plt
from tqdm import tqdm


#from gdpc import __url__, Editor, Block

#editor = Editor(buffering=True)

#load minecraft block array
sub_grid_init = np.loadtxt("subsurface_2.csv",delimiter=" ", dtype=object)

#reshape array to original shape (same as subsurface pull)
sub_grid = sub_grid_init.reshape((200, 200, 44))

In [ ]:
#make a list of all the different types of minecraft blocks in the array
all_blocks = np.unique(sub_grid)
print(all_blocks)

## Making density array

Here we create a density array of the same size as the Minecraft block array. We do this by generating an empty density array of the same size, and assigning a density value to each Miencraft block type present in the original array. Then, we add these density values into the density array wherever we have ore bodies (or whatever items/materials we're looking for in the subsurface). We don't include the density values of the ground (dirt and stone) so that we can just highlight the gravitational response of the materials we're looking for.

In [ ]:
#make an array of zeros the same size as the block array
density_grid = np.zeros(shape = (200, 200, 44))

#wherever we have ore bodies in the block array, insert the corresponding density value in the density array
for x in range(0,200):
    for z in range(0,200):
        for y in range(0,44):
            if sub_grid[x,z,y] == "minecraft:iron_ore":
                density_grid[x,z,y] = 6000
            if sub_grid[x,z,y] == "minecraft:gold_ore":
                density_grid[x,z,y] = 7000
            if sub_grid[x,z,y] == "minecraft:diamond_ore":
                density_grid[x,z,y] = 6000


## Plotting the density

If you want to see where the dense materials are in the subsurface, these cells sum up the density value of each lateral coordinate along the vertical direction, so we can plot a map of total density.

In [ ]:
ore_id = np.sum(density_grid, axis=2)

In [ ]:
fig = plt.figure(figsize=(6,5))
plt.imshow(ore_id,origin='lower')
#plt.contourf(x_obs, y_obs, gz_mgal, 50, cmap='hsv')
#plt.colorbar(label='Gravity anomaly (mGal)')
plt.xlabel('X (km)')
plt.ylabel('Z (km)')
plt.title('Minecraft World Ore Locations')
plt.show()

We can also look at a cross section of the density array to see how deep the different ore bodies are.

In [ ]:
fig = plt.figure(figsize=(6,5))
plt.imshow(density_grid[9,:,:].T)
plt.xlabel('X (km)')
plt.ylabel('Y (km)')
plt.title('Minecraft World Density Cross Section')
plt.show()

## Calculating Gravity Data

Now we can use the density array and the gravity equation to calculate the gravity of the play area.

In [ ]:
G = 6.674e-11  # gravitational constant, m^3/kg/s^2

nx, ny, nz = 200, 200, 44 #number of cells
dx = dy = dz = 1  # cell size in meters

nx_obs, ny_obs, nz_obs = 200, 200, 1 #number of observation points
dx_obs = dy_obs = 1 # width of observation points

# Coordinates of cell centers
x = (np.arange(nx)) * dx + 0.5
y = (np.arange(ny)) * dy + 0.5
z = np.arange(0, -nz, -dz) - 0.5  # depth increasing downward

X, Y, Z = np.meshgrid(x, y, z, indexing='ij')

# Observation grid
x_o = (np.arange(nx_obs)) * dx_obs + 0.5
y_o = (np.arange(ny_obs)) * dy_obs + 0.5
x_obs, y_obs = np.meshgrid(x_o, y_o, indexing='ij')
z_obs = 0.0

# Compute gravity at each observation point
gz = np.zeros_like(x_obs,dtype=float)
for i in range(nx):
    for j in range(ny):
        for k in range(nz):
           rx = x_obs - X[i,j,k]
           ry = y_obs - Y[i,j,k]
           rz = 0 - Z[i,j,k]
           r3 = (rx**2 + ry**2 + rz**2)**1.5
           gz += G * density_grid[i,j,k] * dz * dy * dx * rz / r3

# Convert to mGal
gz_mgal = gz * 1e8

## Plotting the gravity data

In [ ]:
plt.contourf(x_obs, y_obs, gz_mgal, 50, origin = 'lower', cmap='hsv')
cbar = plt.colorbar()
cbar.set_label('Gravity anomaly (mGal)', size = 16)
cbar.ax.tick_params(labelsize=15)
plt.xlabel('x (m)', fontsize = 16)
plt.ylabel('y (m)', fontsize = 16)
plt.tick_params(axis='both', labelsize=14)
plt.savefig('gravity_data_2.png', bbox_inches = 'tight')

In [ ]:
#save the data if you might need it again
np.savetxt('grav_grid.txt',gz_mgal, delimiter=' ')

In [ ]:
#check max and min for building a colour bar for putting the map in the Miencraft world later
np.max(gz_mgal)